In [3]:
import glob, os, subprocess, sys, time
from pathlib import Path

def sh(cmd, check=False):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise RuntimeError(f"failed: {cmd}")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv; free -g | head -2")

wheels = Path(glob.glob("/kaggle/input/**/transformers-*.whl", recursive=True)[0]).parent
sh(f"{sys.executable} -m pip install -q --no-index --find-links {wheels} --no-warn-conflicts transformers==5.18.0", check=True)

code = Path(glob.glob("/kaggle/input/**/run_reap.py", recursive=True)[0]).parent
model = next(Path(p).parent for p in glob.glob("/kaggle/input/**/model.safetensors.index.json", recursive=True)
             if "intel" in p.lower())
traces = {}
for pattern, source in (("*arc-agi-3-milestone-2-solution*", "kaggle_v3"), ("*openrouter*", "openrouter")):
    for f in glob.glob(f"/kaggle/input/**/{pattern}/**/*_requests.jsonl*", recursive=True):
        traces.setdefault(source, str(Path(f).parent))
print("code:", code, "\nmodel:", model, "\ntraces:", traces)
assert len(traces) == 2, "attach the dfranzen notebook output and lamdang/arc3-openrouter-traces"

out = "/kaggle/working/reap/smoke"
t = time.time()
sh(f"{sys.executable} {code}/run_reap.py --model-dir {model} "
   + " ".join(f"--traces {s}={d}" for s, d in traces.items())
   + f" --out {out} --samples kaggle_v3/sk48_p0:0,kaggle_v3/sk48_p0:1,openrouter/ft09_p0:0 --chunk 8192 --precache")
print(f"took {(time.time() - t) / 60:.1f} min")
sh(f"{sys.executable} {code}/analyze.py {out}")

$ nvidia-smi --query-gpu=name,memory.total --format=csv; free -g | head -2
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
               total        used        free      shared  buff/cache   available
Mem:             176           2         169           0           5         174
$ /usr/bin/python3 -m pip install -q --no-index --find-links /kaggle/input/datasets/lamdang/reap-flash-next-wheels --no-warn-conflicts transformers==5.18.0
code: /kaggle/input/datasets/lamdang/reap-flash-next-code 
model: /kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1 
traces: {'kaggle_v3': '/kaggle/input/notebooks/dfranzen/arc-agi-3-milestone-2-solution', 'openrouter': '/kaggle/input/datasets/lamdang/arc3-openrouter-traces'}
$ /usr/bin/python3 /kaggle/input/datasets/lamdang/reap-flash-next-code/run_reap.py --model-dir /kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1 --traces kaggle_

In [4]:
import shutil
from IPython.display import FileLink, display

archive = shutil.make_archive("/kaggle/working/reap_smoke", "zip", root_dir="/kaggle/working", base_dir="reap")
print(archive)
display(FileLink("reap_smoke.zip"))

/kaggle/working/reap_smoke.zip


/kaggle/working/reap_smoke.zip

In [6]:
import glob, sys, time
from pathlib import Path
wheels = Path(glob.glob("/kaggle/input/**/fla_core-*.whl", recursive=True)[0]).parent
code = Path(glob.glob("/kaggle/input/**/prune_eval.py", recursive=True)[0]).parent
sh(f"{sys.executable} -m pip install -q --no-index --find-links {wheels} --no-deps --no-warn-conflicts "
   "flash-linear-attention==0.5.2 fla-core==0.5.2 einops", check=True)

out = "/kaggle/working/reap/prune_eval"
t = time.time()
sh(f"{sys.executable} {code}/prune_eval.py --model-dir {model} --traces kaggle_v3={traces['kaggle_v3']} "
   f"--out {out} --stats-dir /kaggle/working/reap/smoke --precache "
   "--calib-games ar25,bp35,cd82,dc22,g50t,re86,tn36,tu93 --calib-passes 0 --calib-max-tokens 65536 "
   "--eval-games ls20,vc33,sb26 --eval-passes 1 --eval-max-tokens 32768 "
   "--keep 448,384,320,288,256,192")
print(f"took {(time.time() - t) / 60:.1f} min")

$ /usr/bin/python3 -m pip install -q --no-index --find-links /kaggle/input/datasets/lamdang/reap-flash-next-wheels --no-deps --no-warn-conflicts flash-linear-attention==0.5.2 fla-core==0.5.2 einops
$ /usr/bin/python3 /kaggle/input/datasets/lamdang/reap-flash-next-code/prune_eval.py --model-dir /kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1 --traces kaggle_v3=/kaggle/input/notebooks/dfranzen/arc-agi-3-milestone-2-solution --out /kaggle/working/reap/prune_eval --stats-dir /kaggle/working/reap/smoke --precache --calib-games ar25,bp35,cd82,dc22,g50t,re86,tn36,tu93 --calib-passes 0 --calib-max-tokens 65536 --eval-games ls20,vc33,sb26 --eval-passes 1 --eval-max-tokens 32768 --keep 448,384,320,288,256,192
16:19:19 [plan] calibration: ['kaggle_v3/ar25_p0', 'kaggle_v3/bp35_p0', 'kaggle_v3/cd82_p0', 'kaggle_v3/dc22_p0', 'kaggle_v3/g50t_p0', 'kaggle_v3/re86_p0', 'kaggle_v3/tn36_p0', 'kaggle_v3/tu93_p0']
16:19:19 [plan] evaluation: ['kaggle_v3/ls20_p1

In [7]:
code = Path(glob.glob("/kaggle/input/**/prune_eval.py", recursive=True)[0]).parent
for name, extra in [("gate", "--criterion gate"), ("gate_norm", "--criterion gate_norm"),
                    ("reap_generated", "--criterion reap --categories generated")]:
    sh(f"{sys.executable} {code}/prune_eval.py --model-dir {model} --traces kaggle_v3={traces['kaggle_v3']} "
       f"--out /kaggle/working/reap/crit_{name} --stats-dir {out} --stats-dir /kaggle/working/reap/smoke "
       f"--eval-games ls20,vc33,sb26 --eval-passes 1 --eval-max-tokens 32768 "
       f"--keep 384,320,256 --noise-check {extra}")

$ /usr/bin/python3 /kaggle/input/datasets/lamdang/reap-flash-next-code/prune_eval.py --model-dir /kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1 --traces kaggle_v3=/kaggle/input/notebooks/dfranzen/arc-agi-3-milestone-2-solution --out /kaggle/working/reap/crit_gate --stats-dir /kaggle/working/reap/prune_eval --stats-dir /kaggle/working/reap/smoke --eval-games ls20,vc33,sb26 --eval-passes 1 --eval-max-tokens 32768 --keep 384,320,256 --noise-check --criterion gate
16:45:51 [plan] calibration: []
16:45:51 [plan] evaluation: ['kaggle_v3/ls20_p1', 'kaggle_v3/sb26_p1', 'kaggle_v3/vc33_p1']
16:45:53 [reap] loaded model-00001-of-00017.safetensors (1/38)
16:45:54 [reap] loaded model-00003-of-00017.safetensors (2/38)
16:45:55 [reap] loaded model-00004-of-00017.safetensors (3/38)
16:45:56 [reap] loaded model-00005-of-00017.safetensors (4/38)
16:45:57 [reap] loaded model-00006-of-00017.safetensors (5/38)
16:45:58 [reap] loaded model-00007-of-00017.safet

In [8]:
import shutil, os
from IPython.display import FileLink
archive = shutil.make_archive("/kaggle/working/reap_results", "zip", "/kaggle/working", "reap")
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
FileLink("reap_results.zip")

/kaggle/working/reap_results.zip 16.5 MB


/kaggle/working/reap_results.zip